# Enhanced GourNet V3: Robust Attention-Gated CNN for Mango Leaf Disease Classification
### Research Prototype & Thesis Improvement
**Author:** 90377Sednaaa  
**Architecture:** 4-Block Scaled CNN (194k params) with GroupNormalization, Spatial Attention Gate, and Global Average Pooling.

---

### Key Improvements Over V1 and V2:
| Metric / Feature | GourNet V1 (Baseline) | GourNet V2 (Ultra-Lean) | **Enhanced GourNet V3 (Proposed)** |
|---|---|---|---|
| **Parameters** | 683,656 | 98,280 (-85.6%) | **194,793 (-71.5% vs V1)** |
| **Conv Channels** | [16, 32, 64, 64] | [16, 32, 64, 64] | **[32, 64, 96, 128] ($2\times$ feature capacity)** |
| **Normalization** | None | GroupNorm (groups=8) | **GroupNorm (groups=8)** |
| **Pooling Transition** | `Flatten()` (589k dense weights) | `GAP()` (Naive unweighted) | **`Spatial Attention Gate` + `GAP()`** |
| **Outdoor Robustness** | Moderate (redundant weights) | Low on uncropped clutter | **High (Attention suppresses background clutter)** |
| **Augmentation** | Flips + Rotations only | Flips + Rotations + Zoom | **Flips + Rotations + Zoom + Contrast + Brightness** |


## 1. Environment Setup & Dependency Imports


In [ ]:
import os
import sys
import math
import random
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Set random seeds for deterministic reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
print(f"Available GPUs:     {gpu_devices if gpu_devices else 'CPU only'}")


## 2. Dataset Discovery (Auto-detects Kaggle or Local Environment)
This section automatically scans `/kaggle/input/` or local working directories to locate the 8 mango leaf disease classes:
`Anthracnose`, `Bacterial Canker`, `Cutting Weevil`, `Die Back`, `Gall Midge`, `Healthy`, `Powdery Mildew`, `Sooty Mould`.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8

CLASS_NAMES = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould",
]

def find_dataset_dir():
    # Potential search roots
    search_paths = [
        pathlib.Path("/kaggle/input"),
        pathlib.Path("./Dataset"),
        pathlib.Path("../Dataset"),
        pathlib.Path("./samples"),
        pathlib.Path("C:/Users/PC/Downloads/archive/Dataset"),
        pathlib.Path.cwd(),
    ]
    
    for base in search_paths:
        if not base.exists():
            continue
        # Search for any directory that has Anthracnose
        for path in base.rglob("*"):
            if path.is_dir() and (path / "Anthracnose").is_dir():
                return path
            # Check if this folder itself has classes
            subdirs = [d.name.lower() for d in path.iterdir() if d.is_dir()]
            if "anthracnose" in subdirs or "healthy" in subdirs:
                return path

    raise FileNotFoundError("Could not automatically locate the dataset directory. Please specify DATA_DIR manually.")

DATA_DIR = find_dataset_dir()
print(f"Found Dataset at: {DATA_DIR}")

# Verify classes
found_classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")])
print(f"Detected {len(found_classes)} class folders:")
for c in found_classes:
    count = len(list((DATA_DIR / c).glob("*.*")))
    print(f"  - {c:20s}: {count:5d} images")


## 3. Stratified Data Pipeline (80% Train, 10% Validation, 10% Held-Out Test)


In [ ]:
# Load training split (80%)
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

# Load validation + test split (20% total -> split 10% val, 10% test)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

val_test_batches = tf.data.experimental.cardinality(val_test_ds).numpy()
val_size = val_test_batches // 2

val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print(f"Dataset split successfully:")
print(f"  Train batches:      {tf.data.experimental.cardinality(train_ds).numpy()} (~80%)")
print(f"  Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()} (~10%)")
print(f"  Test batches:       {tf.data.experimental.cardinality(test_ds).numpy()} (~10%)")


## 4. Visualizing Sample Leaf Specimens & Data Augmentation Pipeline


In [ ]:
# Visualize raw training samples
plt.figure(figsize=(12, 10))
for images, labels in raw_train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        lbl_idx = int(labels[i].numpy())
        plt.title(found_classes[lbl_idx], fontsize=11, fontweight="bold")
        plt.axis("off")
plt.suptitle("Raw Training Dataset Specimens", fontsize=14, y=0.92)
plt.show()


## 5. Enhanced GourNet V3 Architecture (Balanced Channels + Spatial Attention Gate)

### Technical Innovations:
1. **Scaled Feature Channels `[32, 64, 96, 128]`**: $2\times$ greater filter diversity than V2, providing feature redundancy for atypical outdoor lighting and foliage textures.
2. **In-the-Wild Augmentation Stack**: Adds `RandomContrast` and `RandomBrightness` along with geometric transformations to ensure real-world outdoor robustness.
3. **Spatial Attention Gate**:
   $$\mathbf{A} = \sigma(\text{Conv}_{1\times 1}(\mathbf{F}))$$
   $$\mathbf{F}_{\text{gated}} = \mathbf{F} \odot \mathbf{A}$$
   Suppresses background pixels (dirt, sky, orchard branches) and boosts the lesion regions prior to Global Average Pooling.
4. **GroupNormalization (`groups=8`)**: Prevents small-batch statistic distortion and stabilizes training.
5. **Ultra-Efficient Footprint**: **~194,793 parameters** (71.5% smaller than baseline V1).


In [ ]:
def build_gournet_v3(num_classes: int = 8, input_shape=(224, 224, 3)) -> keras.Model:
    rescale = tf.keras.Sequential([
        layers.Rescaling(1.0 / 255)
    ], name="Sequential-1")

    # In-the-wild robustness data augmentation
    augment = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.20),
        layers.RandomZoom(0.20),
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.20),
    ], name="Sequential-2")

    inputs = layers.Input(shape=input_shape, name="input_specimen")
    x = rescale(inputs)
    x = augment(x)

    # --- Conv Block 1 ---
    x = layers.Conv2D(32, kernel_size=3, padding="same", use_bias=False, name="Block1_Conv")(x)
    x = layers.GroupNormalization(groups=8, name="Block1_GN")(x)
    x = layers.Activation("relu", name="Block1_ReLU")(x)
    x = layers.MaxPooling2D(pool_size=2, name="Block1_Pool")(x)

    # --- Conv Block 2 ---
    x = layers.Conv2D(64, kernel_size=3, padding="same", use_bias=False, name="Block2_Conv")(x)
    x = layers.GroupNormalization(groups=8, name="Block2_GN")(x)
    x = layers.Activation("relu", name="Block2_ReLU")(x)
    x = layers.MaxPooling2D(pool_size=2, name="Block2_Pool")(x)

    # --- Conv Block 3 ---
    x = layers.Conv2D(96, kernel_size=3, padding="same", use_bias=False, name="Block3_Conv")(x)
    x = layers.GroupNormalization(groups=8, name="Block3_GN")(x)
    x = layers.Activation("relu", name="Block3_ReLU")(x)
    x = layers.MaxPooling2D(pool_size=2, name="Block3_Pool")(x)

    # --- Conv Block 4 (Grad-CAM Target Layer) ---
    x = layers.Conv2D(128, kernel_size=3, padding="same", use_bias=False, name="Block4_Conv")(x)
    x = layers.GroupNormalization(groups=8, name="Block4_GN")(x)
    x = layers.Activation("relu", name="Block4_ReLU")(x)
    x = layers.MaxPooling2D(pool_size=2, name="Block4_Pool")(x)

    # Spatial Regularization
    x = layers.SpatialDropout2D(0.20, name="Spatial_Dropout")(x)

    # --- Spatial Attention Gate ---
    # Computes a 1-channel spatial saliency mask (values between 0.0 and 1.0)
    attention_mask = layers.Conv2D(1, kernel_size=1, activation="sigmoid", name="Spatial_Attention_Gate")(x)
    # Background gets multiplied by ~0, salient lesion features multiplied by ~1
    gated_features = layers.Multiply(name="Attention_Multiply")([x, attention_mask])

    # Global Average Pooling across the gated features (avoids background dilution)
    pooled = layers.GlobalAveragePooling2D(name="GAP")(gated_features)

    # Dense Classification Head
    dense = layers.Dense(64, use_bias=False, name="Dense-1")(pooled)
    dense = layers.GroupNormalization(groups=8, name="Dense-1_GN")(dense)
    dense = layers.Activation("relu", name="Dense-1_ReLU")(dense)
    dense = layers.Dropout(0.40, name="Head_Dropout")(dense)

    outputs = layers.Dense(num_classes, activation="softmax", name="Dense-2")(dense)

    model = keras.Model(inputs=inputs, outputs=outputs, name="GourNet_V3_Robust")
    return model

model_v3 = build_gournet_v3(num_classes=NUM_CLASSES)
model_v3.summary()
print(f"\nTotal Parameters: {model_v3.count_params():,}")


## 6. Training with Learning Rate Decay & Early Stopping


In [ ]:
LEARNING_RATE = 0.001
MAX_EPOCHS = 60
EARLY_STOP_PATIENCE = 7

model_v3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOP_PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

checkpoint = callbacks.ModelCheckpoint(
    "gournet_v3_robust_best.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1,
)

print("Starting GourNet V3 training...")
history_v3 = model_v3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=[early_stopping, checkpoint, reduce_lr],
)
print("Training completed!")


## 7. Training & Validation Trends Dashboard
Visualizes training vs validation accuracy, loss trajectories, best epoch checkpointing, and learning rate decay over time.


In [ ]:
acc = history_v3.history["accuracy"]
val_acc = history_v3.history["val_accuracy"]
loss = history_v3.history["loss"]
val_loss = history_v3.history["val_loss"]
epochs_range = range(1, len(acc) + 1)
best_epoch = int(np.argmin(val_loss)) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Accuracy Trends
axes[0].plot(epochs_range, [a * 100 for a in acc], 'o-', label="Training Accuracy", color="#1976D2", linewidth=2, markersize=4)
axes[0].plot(epochs_range, [va * 100 for va in val_acc], 's-', label="Validation Accuracy", color="#388E3C", linewidth=2, markersize=4)
axes[0].axvline(best_epoch, color="#D32F2F", linestyle="--", alpha=0.8, label=f"Best Model (Epoch {best_epoch})")
axes[0].scatter(best_epoch, val_acc[best_epoch-1]*100, color="#D32F2F", s=100, zorder=5)
axes[0].set_title("Training vs Validation Accuracy", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Accuracy (%)", fontsize=11)
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend(loc="lower right")

# 2. Loss Trends
axes[1].plot(epochs_range, loss, 'o-', label="Training Loss", color="#1976D2", linewidth=2, markersize=4)
axes[1].plot(epochs_range, val_loss, 's-', label="Validation Loss", color="#E65100", linewidth=2, markersize=4)
axes[1].axvline(best_epoch, color="#D32F2F", linestyle="--", alpha=0.8, label=f"Min Loss ({val_loss[best_epoch-1]:.4f})")
axes[1].scatter(best_epoch, val_loss[best_epoch-1], color="#D32F2F", s=100, zorder=5)
axes[1].set_title("Training vs Validation Loss", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Cross-Entropy Loss", fontsize=11)
axes[1].grid(True, linestyle=":", alpha=0.6)
axes[1].legend(loc="upper right")

# 3. Generalization Gap (Overfitting Monitor)
gen_gap = [abs(t - v) for t, v in zip(loss, val_loss)]
axes[2].plot(epochs_range, gen_gap, 'd-', color="#7B1FA2", linewidth=2, markersize=4, label="|Train Loss - Val Loss|")
axes[2].axvline(best_epoch, color="#D32F2F", linestyle="--", alpha=0.8, label=f"Best Epoch Checkpoint")
axes[2].set_title("Generalization Gap Trend", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch", fontsize=11)
axes[2].set_ylabel("Absolute Loss Difference", fontsize=11)
axes[2].grid(True, linestyle=":", alpha=0.6)
axes[2].legend(loc="upper left")

plt.tight_layout()
plt.savefig("gournet_v3_training_validation_trends.png", dpi=300)
plt.show()

print(f"\nBest Model Summary:")
print(f"  Best Epoch:           {best_epoch}")
print(f"  Best Val Accuracy:    {val_acc[best_epoch-1]*100:.2f}%")
print(f"  Lowest Val Loss:      {val_loss[best_epoch-1]:.4f}")


## 8. Held-Out Test Set Evaluation & Dual Confusion Matrices (Raw & Normalized)


In [ ]:
test_loss, test_acc = model_v3.evaluate(test_ds, verbose=1)
print(f"\n=======================================================")
print(f"HELD-OUT TEST SET ACCURACY: {test_acc * 100:.2f}%")
print(f"HELD-OUT TEST SET LOSS:     {test_loss:.4f}")
print(f"=======================================================")

# Generate predictions across test set
y_true = []
y_pred = []
y_probs = []

for images, labels in test_ds:
    preds = model_v3.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

from sklearn.metrics import confusion_matrix, classification_report

cm_raw = confusion_matrix(y_true, y_pred)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis] * 100.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# 1. Raw counts
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", xticklabels=found_classes, yticklabels=found_classes, cbar=True, ax=ax1)
ax1.set_title("Test Set Confusion Matrix (Raw Counts)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Predicted Label", fontsize=10)
ax1.set_ylabel("True Label", fontsize=10)
ax1.tick_params(axis='x', rotation=45)

# 2. Normalized percentages
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", xticklabels=found_classes, yticklabels=found_classes, cbar=True, ax=ax2)
ax2.set_title("Test Set Confusion Matrix (Normalized % / Sensitivity)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Predicted Label", fontsize=10)
ax2.set_ylabel("True Label", fontsize=10)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("gournet_v3_confusion_matrices.png", dpi=300)
plt.show()

# Classification Report
print("\n--- Detailed Classification Metrics ---")
report = classification_report(y_true, y_pred, target_names=found_classes, digits=4)
print(report)


## 9. Per-Class Precision, Recall, and F1-Score Breakdown


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=range(len(found_classes)))

metrics_df = pd.DataFrame({
    "Class": found_classes,
    "Precision": precision * 100,
    "Recall (Sensitivity)": recall * 100,
    "F1-Score": f1 * 100,
    "Test Support": support,
})

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(found_classes))
width = 0.25

rects1 = ax.bar(x - width, metrics_df["Precision"], width, label="Precision (%)", color="#1E88E5")
rects2 = ax.bar(x, metrics_df["Recall (Sensitivity)"], width, label="Recall (%)", color="#43A047")
rects3 = ax.bar(x + width, metrics_df["F1-Score"], width, label="F1-Score (%)", color="#FB8C00")

ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("Per-Class Disease Detection Performance Metrics (GourNet V3)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(found_classes, rotation=30, ha="right", fontsize=10)
ax.set_ylim(80, 103)
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig("gournet_v3_per_class_metrics.png", dpi=300)
plt.show()
display(metrics_df)


## 10. Visualizing the Spatial Attention Gate & Grad-CAM Heatmaps
Demonstrates how the **Spatial Attention Gate** suppresses background clutter (dirt, branches, sky) and focuses directly on the disease lesions.


In [ ]:
# Attention extractor sub-model
attention_submodel = keras.Model(
    inputs=model_v3.input,
    outputs=[
        model_v3.get_layer("Block4_ReLU").output,
        model_v3.get_layer("Spatial_Attention_Gate").output,
        model_v3.output,
    ]
)

def visualize_sample_attention(test_dataset, num_samples=4):
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    sample_count = 0
    
    for images, labels in test_dataset:
        for idx in range(images.shape[0]):
            if sample_count >= num_samples:
                break
                
            img_tensor = tf.expand_dims(images[idx], axis=0)
            orig_img = images[idx].numpy().astype("uint8")
            true_cls = found_classes[int(labels[idx].numpy())]
            
            with tf.GradientTape() as tape:
                conv_out, attn_map, preds = attention_submodel(img_tensor)
                pred_idx = tf.argmax(preds[0])
                pred_cls = found_classes[int(pred_idx.numpy())]
                conf = float(preds[0][pred_idx])
                loss = preds[:, pred_idx]
                
            grads = tape.gradient(loss, conv_out)
            pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
            cam = tf.reduce_sum(conv_out[0] * pooled_grads, axis=-1).numpy()
            cam = np.maximum(cam, 0)
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            cam = cv2_cam = np.array(Image.fromarray(cam).resize((224, 224), Image.Resampling.BILINEAR))
            
            attn_2d = attn_map[0, :, :, 0].numpy()
            attn_resized = np.array(Image.fromarray(attn_2d).resize((224, 224), Image.Resampling.BILINEAR))
            
            # Subplot 1: Original
            axes[sample_count, 0].imshow(orig_img)
            axes[sample_count, 0].set_title(f"True: {true_cls}", fontsize=10, fontweight="bold")
            axes[sample_count, 0].axis("off")
            
            # Subplot 2: Spatial Attention Gate
            axes[sample_count, 1].imshow(attn_resized, cmap="magma")
            axes[sample_count, 1].set_title("Learned Spatial Attention", fontsize=10)
            axes[sample_count, 1].axis("off")
            
            # Subplot 3: Grad-CAM Activation
            axes[sample_count, 2].imshow(cam, cmap="jet")
            axes[sample_count, 2].set_title("Grad-CAM Feature Map", fontsize=10)
            axes[sample_count, 2].axis("off")
            
            # Subplot 4: Overlay
            overlay = np.float32(orig_img) / 255.0
            heatmap_color = plt.cm.jet(cam)[:, :, :3]
            blended = 0.6 * overlay + 0.4 * heatmap_color
            axes[sample_count, 3].imshow(np.clip(blended, 0, 1))
            axes[sample_count, 3].set_title(f"Pred: {pred_cls} ({conf*100:.1f}%)", fontsize=10, color="green" if pred_cls == true_cls else "red")
            axes[sample_count, 3].axis("off")
            
            sample_count += 1
        if sample_count >= num_samples:
            break
            
    plt.tight_layout()
    plt.savefig("gournet_v3_attention_gradcam_gallery.png", dpi=300)
    plt.show()

visualize_sample_attention(test_ds, num_samples=4)


## 11. Model Export for Web Deployment & Streamlit Prototype
Exports the trained GourNet V3 model in both modern Keras 3 format (`.keras`) and weights/SavedModel directory for seamless pairing with the Streamlit comparative prototype.


In [ ]:
# Save best model to working directory
model_v3.save("gournet_v3_robust_final.keras")
model_v3.save_weights("gournet_v3_robust.weights.h5")

# Also export as directory format for cross-platform inference
export_dir = pathlib.Path("gournet_v3_model")
model_v3.save(str(export_dir))

print("Model successfully exported:")
print(f"  1. gournet_v3_robust_final.keras ({os.path.getsize('gournet_v3_robust_final.keras') / 1024 / 1024:.2f} MB)")
print(f"  2. gournet_v3_robust.weights.h5  ({os.path.getsize('gournet_v3_robust.weights.h5') / 1024 / 1024:.2f} MB)")
print(f"  3. gournet_v3_model/ (SavedModel Directory)")
print("\nDownload these artifacts from Kaggle Output to use them in your thesis Streamlit application!")
